In [ ]:
from transformers import AutoModel
from torch.optim.lr_scheduler import CosineAnnealingLR
from datasets import load_dataset
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch import nn, Tensor
import pandas as pd
from torchmetrics.classification import MultilabelAveragePrecision
import torch, dotenv, os
from utils import transform_train, transform_test
from model import Tagger
from torch.utils.tensorboard import SummaryWriter

dotenv.load_dotenv()
token = os.getenv('HF_TOKEN')

pd.set_option('display.float_format', '{:.2f}'.format)

In [2]:
to_predict = (
    'female', 'male',
    'unicorn', 'pegasus', 'earth pony', 'alicorn',
    'simple background', 'monochrome',
    'clothes', 'wings', 'horn', 'chest fluff', 'ear fluff', 'hat', 'jewelry', 'food', 'foal',
    'looking at you', 'smiling', 'open mouth', 'blushing', 'sitting', 'raised hoof', 'eyes closed',
    'twilight sparkle', 'fluttershy', 'rainbow dash', 'pinkie pie', 'rarity', 'applejack'
)

In [3]:
ds = load_dataset('Brambles/Ponies', token=token)

README.md:   0%|          | 0.00/1.31k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

data/train-00000-of-00031.parquet:   0%|          | 0.00/929M [00:00<?, ?B/s]

data/train-00001-of-00031.parquet:   0%|          | 0.00/910M [00:00<?, ?B/s]

...

data/test-00002-of-00003.parquet:   0%|          | 0.00/790M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/174407 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10593 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11929 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/57 [00:00<?, ?it/s]

In [ ]:
epochs = 5
batch_size = 155
lr = 0.001
weight_decay = 0.001
temp_alpha = 1.0
mix_alpha = 0.3
gamma = 0.005

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [5]:
def t_train(batch):
    return {
        'images': torch.stack([transform_train(img) for img in batch['image']]),
        'tags': torch.tensor(batch['tags'], dtype=torch.float32)
    }

def t_test(batch):
    return {
        'images': torch.stack([transform_test(img) for img in batch['image']]),
        'tags': torch.tensor(batch['tags'], dtype=torch.float32)
    }

In [6]:
dl_train = DataLoader(
    ds['train'].with_transform(t_train),
    batch_size=batch_size,
    pin_memory=True,
    num_workers=4,
    persistent_workers=True,
    shuffle=True,
    drop_last=True
)

dl_test = DataLoader(
    ds['test'].with_transform(t_test),
    batch_size=batch_size,
    pin_memory=True,
    num_workers=3,
    persistent_workers=True
)

In [7]:
model = Tagger().to(device)

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

In [8]:
optim = AdamW([
        { 'params': model.backbone.parameters(), 'lr': lr * 0.06 },
        { 'params': model.classifier.parameters(), 'lr': lr },
    ],
    weight_decay=weight_decay
)

scheduler = CosineAnnealingLR(optim, epochs)

In [9]:
def vpu_loss(logits: Tensor, labels: Tensor):
    temp = (temp_alpha * logits.std(0, False)).clamp(0.05, 1.0).detach()

    outputs = (logits / temp).sigmoid()
    log_pos_probs = outputs.clamp_min(1e-10).log() * labels

    num_p_rec = 1 / labels.sum(0)
    num_p_rec = torch.where(num_p_rec.isinf(), torch.zeros_like(num_p_rec), num_p_rec)
    num_u_rec = 1 / labels.shape[0]

    p_c = outputs.mean(0)
    u_loss = (num_u_rec * outputs.sum(0)).clamp_min(1e-10).log()
    p_loss = num_p_rec * log_pos_probs.sum(0)

    class_losses = p_c.pow(gamma).detach() * u_loss - p_loss

    return class_losses.sum()


beta_dist = torch.distributions.Beta(mix_alpha, mix_alpha)

def mixup_loss(x: Tensor, preds: Tensor, y: Tensor, model):
    midpoint1 = int(x.shape[0] / 2)
    # This makes sure all partitions are the same size
    midpoint2 = midpoint1 + (1 if x.shape[0] % 2 == 1 else 0)

    x1, x2 = x[:midpoint1], x[midpoint2:] # logits
    y1, y2 = y[:midpoint1], y[midpoint2:] # labels
    yh1, yh2 = preds[:midpoint1], preds[midpoint2:] # preds

    target1 = y1 + yh1 * (y1 == 0)
    target2 = y2 + yh2 * (y2 == 0)

    idx_perm = torch.randperm(len(x1))

    lam = beta_dist.sample()

    x_mixed = x1[idx_perm]      * lam + x2      * (1 - lam)
    y_mixed = target1[idx_perm] * lam + target2 * (1 - lam)

    with torch.autocast('cuda', torch.bfloat16):
        outputs = model(x_mixed).float()

    log_outputs = outputs.sigmoid().clamp_min(1e-10).log().float()

    return (((y_mixed.clamp_min(1e-10).log() - log_outputs).pow(2).sum(dim=0)) * (1 / len(x1))).sum()

In [10]:
f_mAP = MultilabelAveragePrecision(len(to_predict))

def evaluate(epoch, prev_best, save_best=True):
    model.eval()
    preds = torch.zeros((len(ds['test']), len(to_predict)))
    test_loss = 0

    with torch.no_grad():
        for i, batch in enumerate(dl_test):
            X: torch.Tensor = batch['images'].to(device, non_blocking=True)
            y: torch.Tensor = batch['tags'].to(device, non_blocking=True)

            logits: torch.Tensor = model(X)
            test_loss += vpu_loss(logits, y)

            preds[i * batch_size:(i + 1) * batch_size] = logits.sigmoid()

        mAP: Tensor = f_mAP(preds.sigmoid(), torch.as_tensor(ds['test']['tags']))

        test_loss = test_loss / len(dl_test)

        print(f'TL: {test_loss:.3} mAP: {mAP:.3f}\n')

        if mAP > prev_best:
            if save_best:
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optim.state_dict(),
                }, f'outputs/Best3-E{epoch}.pt')

            return mAP

        return prev_best

In [ ]:
writer = SummaryWriter()
epoch_steps = len(ds) // batch_size

In [13]:
def train_epoch(epoch):
    model.train()
    train_loss = 0

    for i, batch in enumerate(dl_train, 1):
        X: torch.Tensor = batch['images'].to(device, non_blocking=True)
        y: torch.Tensor = batch['tags'].to(device, non_blocking=True)

        with torch.autocast('cuda', torch.bfloat16):
            logits: Tensor = model(X).float()

        preds = logits.sigmoid()
        loss = vpu_loss(logits, y) + mixup_loss(X, preds, y, model)

        loss_delta = loss.item()
        train_loss += loss_delta

        loss.backward()
        optim.step()
        scheduler.step()
        optim.zero_grad()

        if (i % 5 == 0):
            #writer.add_scalar('Loss/train', loss_delta, epoch * epoch_steps + i)
            print(f'{' '*40}\r{epoch}/{epochs}: %{i * 100 / len(dl_train):.2f}  {train_loss:.2f}  d{loss_delta:.2f}  Avg conf: {preds.sum() * 100 / preds.numel():.2f}%', end='\r')

    print(f'\nAverage Loss: {train_loss / len(dl_train):.3f}{' '*50}')
    return evaluate(epoch, best_mAP)

In [14]:
best_mAP = 0

for epoch in range(1, epochs + 1):
    model.freeze_backbone(epoch == 1)
    best_mAP = train_epoch(epoch)


1/5: %100.00  -13067.09  d-16.42  Avg conf: 39.22%
Average Loss: -11.615                                                  
TL: -19.8 mAP: 0.507

2/5: %100.00  -28417.00  d-33.31  Avg conf: 29.26%
Average Loss: -25.260                                                  
TL: -35.0 mAP: 0.674

3/5: %100.00  -35040.74  d-35.95  Avg conf: 28.19%
Average Loss: -31.147                                                  
TL: -40.4 mAP: 0.694

4/5: %100.00  -37324.00  d-36.88  Avg conf: 26.23%
Average Loss: -33.177                                                  
TL: -42.5 mAP: 0.705

5/5: %100.00  -39150.50  d-37.56  Avg conf: 27.48%
Average Loss: -34.800                                                  
TL: -42.7 mAP: 0.710

